# Twin-XGB Boosted v2 — Tail Alpha 200 + Dynamic Quantile

**Optimasi dari v1:**
1. **ALPHA_UNDER = 200** (was 50) — mengurangi under-forecast di tail items
2. **THRESHOLD_GRID = 0.05–0.90** (was 0.10) — threshold optimal lebih rendah
3. **Dynamic quantile** — hari dengan holiday_intensity > 2× pakai q=0.95, sisanya q=0.90
4. **Holiday intensity outlier analysis** — filter country dengan <5 holiday


In [ ]:
# ============================================================
# [KAGGLE SETUP] Install dependencies + GPU detection + imports
# ============================================================
import importlib, sys, os, warnings, json, gc, time
from pathlib import Path
import numpy as np
import pandas as pd

# Install holidays (not pre-installed on Kaggle)
_start = time.time()
try:
    importlib.import_module('holidays')
    print("[OK] holidays already installed")
except ImportError:
    print("[INSTALL] Installing holidays...")
    !pip install holidays -q
    print("[OK] holidays installed")

import holidays
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, f1_score, precision_score, recall_score
from sklearn.calibration import CalibratedClassifierCV
import matplotlib.pyplot as plt
import xgboost as xgb

# ── GPU Detection ──
print("[INFO] XGBoost version:", xgb.__version__)
try:
    import torch
    cuda_ok = torch.cuda.is_available()
except ImportError:
    cuda_ok = False
try:
    _ = xgb.XGBRegressor(n_estimators=1, device='cuda')
    DEVICE = 'cuda'
    print(f"[OK] GPU device=cuda: SUPPORTED")
except Exception:
    try:
        _ = xgb.XGBRegressor(n_estimators=1, tree_method='gpu_hist')
        DEVICE = 'cuda'
        print(f"[OK] GPU tree_method=gpu_hist: SUPPORTED")
    except Exception as e:
        DEVICE = 'cpu'
        print(f"[WARN] GPU not supported, falling back to CPU: {e}")

print(f"[OK] Using device={DEVICE} ({time.time()-_start:.1f}s)")

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 140)
pd.set_option('mode.copy_on_write', True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
# ============================================================
# [CONFIG] Paths, constants, non-product codes
# ============================================================
RAW_PATH = "/kaggle/input/datasets/wildanmaulana1/online-retail-fmcg/online_retail.csv"
CACHE_DIR = Path("/kaggle/working")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_PATH = str(CACHE_DIR / "panel_cached.parquet")
FULL_CACHE = str(CACHE_DIR / "panel_full.parquet")

print(f"[CONFIG] RAW_PATH: {RAW_PATH}")
print(f"[CONFIG] CACHE_PATH: {CACHE_PATH}")
print(f"[CONFIG] FULL_CACHE: {FULL_CACHE}")
print(f"[CONFIG] RAW_PATH exists: {Path(RAW_PATH).exists()}")
if not Path(RAW_PATH).exists():
    parent = Path(RAW_PATH).parent
    print(f"[WARN] RAW_PATH missing. Listing parent: {parent}")
    if parent.exists():
        for p in parent.iterdir():
            print(f"  - {p}")

CHUNK_SIZE = 200_000
USECOLS = ['Invoice', 'StockCode', 'Quantity', 'InvoiceDate', 'Price', 'Country']
DTYPES = {
    'Invoice': 'string', 'StockCode': 'string', 'Quantity': 'float32',
    'Price': 'float32', 'Country': 'string',
}
# Configurable caps & quantiles
HOLIDAY_INTENSITY_CAP = 100.0  # clip holiday intensity ratio
QUANTILE_Q_EXTREME = 0.98  # quantile for peak+holiday overlap

NON_PRODUCT_CODES = {
    'POST', 'DOT', 'C2', 'M', 'D', 'ADJUST', 'ADJUST2',
    'BANK CHARGES', 'AMAZONFEE', 'B', 'S', 'PADS',
    'TEST001', 'TEST002', 'GIFT_0001_10', 'GIFT_0001_20',
    'GIFT_0001_30', 'GIFT_0001_40', 'GIFT_0001_50', 'GIFT_0001_70',
    'GIFT_0001_80',
}



In [ ]:
# ============================================================
# [PREPROCESSING] Chunk loading → daily aggregation
# ============================================================
def preprocess_chunk(chunk):
    _t = time.time()
    df = chunk.rename(
        columns={'Invoice': 'invoice', 'StockCode': 'stock_code',
                 'Quantity': 'quantity', 'InvoiceDate': 'invoice_date',
                 'Price': 'price', 'Country': 'country'}).copy()
    df['invoice_date'] = pd.to_datetime(df['invoice_date'], errors='coerce')
    df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce').astype('float32')
    df['price'] = pd.to_numeric(df['price'], errors='coerce').astype('float32')
    df['stock_code'] = df['stock_code'].astype('string').str.strip().str.upper()
    df['invoice'] = df['invoice'].astype('string').str.strip()
    df['country'] = df['country'].astype('string').str.strip()
    df = df.drop_duplicates()
    df = df[df['invoice_date'].notna()]
    df = df[df['stock_code'].notna() & (df['stock_code'] != '')]
    df = df[df['invoice'].notna() & (df['invoice'] != '')]
    df = df[~df['invoice'].str.startswith('C', na=False)]
    df = df[~df['stock_code'].isin(NON_PRODUCT_CODES)]
    df = df[(df['quantity'] > 0) & (df['price'] > 0)]
    df['date'] = df['invoice_date'].dt.normalize()
    df['revenue'] = df['quantity'] * df['price']
    df['stock_code'] = df['stock_code'].astype('category')
    df['country'] = df['country'].astype('category')
    df['invoice'] = df['invoice'].astype('category')
    print(f"  [PREPROCESS] chunk processed: {len(df)} rows in {time.time()-_t:.1f}s")
    return df[['stock_code', 'country', 'date', 'invoice', 'quantity', 'price', 'revenue']]

def aggregate_daily_from_chunks(path):
    _t0 = time.time()
    parts = []
    max_parts = 25
    chunk_i = 0
    for chunk in pd.read_csv(path, usecols=USECOLS, dtype=DTYPES, chunksize=CHUNK_SIZE, low_memory=False):
        chunk_i += 1
        print(f"  [CHUNK {chunk_i}] Reading...")
        cleaned = preprocess_chunk(chunk)
        daily = cleaned.groupby(['stock_code', 'country', 'date'], as_index=False, observed=True).agg(
            demand_qty=('quantity', 'sum'), revenue=('revenue', 'sum'),
            num_invoices=('invoice', 'nunique'), price_mean=('price', 'mean'))
        parts.append(daily)
        if len(parts) >= max_parts:
            partial = pd.concat(parts, ignore_index=True)
            parts = [partial.groupby(['stock_code', 'country', 'date'], as_index=False, observed=True)
                     .agg(demand_qty=('demand_qty', 'sum'), revenue=('revenue', 'sum'),
                          num_invoices=('num_invoices', 'sum'), price_mean=('price_mean', 'mean'))]
            del partial; gc.collect()
        del chunk, cleaned, daily; gc.collect()
    daily_all = pd.concat(parts, ignore_index=True)
    del parts; gc.collect()
    daily_all = daily_all.groupby(['stock_code', 'country', 'date'], as_index=False, observed=True).agg(
        demand_qty=('demand_qty', 'sum'), revenue=('revenue', 'sum'),
        num_invoices=('num_invoices', 'sum'), price_mean=('price_mean', 'mean'))
    daily_all['avg_price'] = np.where(daily_all['demand_qty'] > 0,
                                      daily_all['revenue'] / daily_all['demand_qty'], daily_all['price_mean'])
    daily_all = daily_all.drop(columns=['price_mean'])
    for c in ['stock_code','country']: daily_all[c] = daily_all[c].astype('category')
    for c in ['demand_qty','revenue','avg_price','num_invoices']: daily_all[c] = daily_all[c].astype('float32')
    print(f"[OK] aggregate_daily done: {daily_all.shape} in {time.time()-_t0:.1f}s")
    return daily_all


In [ ]:
# ============================================================
# [LOAD DATA] Build panel with zero-filled daily grid
# ============================================================
_t0 = time.time()

def build_full_daily_panel(df):
    df = df.sort_values(['stock_code', 'country', 'date']).reset_index(drop=True)
    def _resample(group):
        group = group.set_index('date').asfreq('D')
        group['stock_code'] = group['stock_code'].iloc[0]
        group['country'] = group['country'].iloc[0]
        return group.reset_index()
    panel = df.groupby(['stock_code', 'country'], group_keys=False, sort=False).apply(_resample)
    panel = panel.reset_index(drop=True)
    for c in ['demand_qty','revenue','num_invoices']: panel[c] = panel[c].fillna(0)
    panel['avg_price'] = panel['avg_price'].astype('float32')
    panel['avg_price'] = panel.groupby(['stock_code', 'country'], sort=False)['avg_price'].ffill().bfill().fillna(0)
    for c in ['stock_code','country']: panel[c] = panel[c].astype('category')
    for c in ['demand_qty','revenue','avg_price','num_invoices']: panel[c] = panel[c].astype('float32')
    return panel

group_cols = ['stock_code', 'country']

if os.path.exists(CACHE_PATH):
    panel_df = pd.read_parquet(CACHE_PATH)
    print(f"[LOAD] Loaded cached panel: {panel_df.shape}")
else:
    print("[LOAD] Building daily aggregation from raw CSV...")
    daily_df = aggregate_daily_from_chunks(RAW_PATH)
    print(f"[LOAD] Building panel (zero-filled daily grid)...")
    panel_df = build_full_daily_panel(daily_df)
    del daily_df; gc.collect()
    panel_df = panel_df.sort_values(['stock_code', 'country', 'date']).reset_index(drop=True)
    panel_df.to_parquet(CACHE_PATH)
    print(f"[OK] Cached panel saved: {panel_df.shape}")

# Filter low-observation items BEFORE heavy feature engineering
MIN_OBS = 60
item_obs = panel_df.groupby(group_cols).size()
panel_df = panel_df[panel_df.set_index(group_cols).index.isin(item_obs[item_obs >= MIN_OBS].index)].copy()
panel_df = panel_df.sort_values(group_cols + ['date']).reset_index(drop=True)
n_items = panel_df[group_cols].drop_duplicates().shape[0]
print(f"[OK] After MIN_OBS={MIN_OBS} filter: {panel_df.shape}, items: {n_items}")
print(f"[TIME] Data loading: {time.time()-_t0:.1f}s")
panel_df.head()


In [ ]:
# ============================================================
# [FEATURE ENGINEERING] Calendar → Holiday → Lag/Rolling → Intensity → Peak
# ============================================================
_t0 = time.time()

if os.path.exists(FULL_CACHE):
    panel_df = pd.read_parquet(FULL_CACHE)
    print(f"[LOAD] Loaded full preprocessed panel: {panel_df.shape}")
else:
    print("[FE] Generating full feature set...")

    # ── Calendar + Holiday (14 features) ──
    COUNTRY_TO_HOLIDAYS = {
        'Australia': 'AU', 'Austria': 'AT', 'Belgium': 'BE', 'Canada': 'CA',
        'Channel Islands': 'GB', 'Czech Republic': 'CZ', 'Denmark': 'DK', 'EIRE': 'IE',
        'Finland': 'FI', 'France': 'FR', 'Germany': 'DE', 'Greece': 'GR',
        'Hong Kong': 'HK', 'Iceland': 'IS', 'Israel': 'IL', 'Italy': 'IT',
        'Japan': 'JP', 'Korea': 'KR', 'Netherlands': 'NL', 'Nigeria': 'NG',
        'Norway': 'NO', 'Poland': 'PL', 'Portugal': 'PT', 'RSA': 'ZA',
        'Saudi Arabia': 'SA', 'Singapore': 'SG', 'Spain': 'ES', 'Sweden': 'SE',
        'Switzerland': 'CH', 'Thailand': 'TH', 'USA': 'US',
        'United Kingdom': 'GB', 'United Arab Emirates': 'AE',
        'European Community': None, 'Unspecified': None, 'West Indies': None,
        'Bahrain': 'BH', 'Bermuda': 'BM', 'Cyprus': 'CY', 'Lithuania': 'LT',
        'Malta': 'MT', 'Lebanon': 'LB',
    }
    panel_df['country_code'] = panel_df['country'].map(COUNTRY_TO_HOLIDAYS).astype('category')
    years = sorted(panel_df['date'].dt.year.unique().tolist())
    supported = set(holidays.list_supported_countries())
    holiday_rows = []
    for code in sorted(panel_df['country_code'].dropna().astype(str).unique()):
        if code not in supported: continue
        hset = holidays.country_holidays(code, years=years)
        holiday_rows.append(pd.DataFrame({'country_code': code, 'date': list(hset.keys()), 'is_hari_besar': 1}))
    hdf = pd.concat(holiday_rows, ignore_index=True) if holiday_rows else pd.DataFrame(columns=['country_code', 'date', 'is_hari_besar'])
    hdf['date'] = pd.to_datetime(hdf['date'], errors='coerce')
    panel_df = panel_df.merge(hdf, on=['country_code', 'date'], how='left', copy=False)
    if 'is_hari_besar' not in panel_df.columns: panel_df['is_hari_besar'] = 0
    panel_df['is_hari_besar'] = panel_df['is_hari_besar'].fillna(0.0).astype('float32').astype('uint8')
    pre_rows = []
    if not hdf.empty:
        for offset in [1, 2, 3]:
            pre_rows.append(hdf.assign(date=hdf['date'] - pd.Timedelta(days=offset), is_pre_hari_besar=1
            )[['country_code', 'date', 'is_pre_hari_besar']])
    pre_df = pd.concat(pre_rows, ignore_index=True) if pre_rows else pd.DataFrame(columns=['country_code', 'date', 'is_pre_hari_besar'])
    pre_df = pre_df.drop_duplicates()
    panel_df = panel_df.merge(pre_df, on=['country_code', 'date'], how='left', copy=False)
    del pre_df, pre_rows, hdf, holiday_rows; gc.collect()
    panel_df['is_pre_hari_besar'] = panel_df['is_pre_hari_besar'].fillna(0.0).astype('float32').astype('uint8')
    panel_df['day_of_week'] = panel_df['date'].dt.dayofweek.astype('int8')
    panel_df['week_of_year'] = panel_df['date'].dt.isocalendar().week.astype('int16')
    panel_df['month'] = panel_df['date'].dt.month.astype('int8')
    panel_df['quarter'] = panel_df['date'].dt.quarter.astype('int8')
    panel_df['day_of_month'] = panel_df['date'].dt.day.astype('int8')
    panel_df['is_weekend'] = (panel_df['day_of_week'] >= 5).astype('uint8')
    panel_df['is_month_start'] = panel_df['date'].dt.is_month_start.astype('uint8')
    panel_df['is_month_end'] = panel_df['date'].dt.is_month_end.astype('uint8')
    panel_df['days_to_month_end'] = ((panel_df['date'] + pd.offsets.MonthEnd(0)) - panel_df['date']).dt.days.astype('int16')
    panel_df['week_of_month'] = ((panel_df['date'].dt.day - 1) // 7 + 1).astype('int8')
    panel_df['is_month_start_window'] = (panel_df['date'].dt.day <= 5).astype('uint8')
    panel_df['is_month_end_window'] = (panel_df['date'].dt.day >= 25).astype('uint8')
    gc.collect()
    print("[FE] Calendar+holiday features done")

    # ── Lag + Rolling (33 features) ──
    panel_df = panel_df.sort_values(group_cols + ['date']).reset_index(drop=True)
    demand_shifted = panel_df.groupby(group_cols)['demand_qty'].shift(1)
    for lag in [1, 2, 7, 14, 21, 28, 35, 56, 84]:
        panel_df[f'demand_lag_{lag}'] = panel_df.groupby(group_cols)['demand_qty'].shift(lag)
    panel_df['roll_max_7'] = demand_shifted.rolling(7, min_periods=1).max().values
    panel_df['roll_max_28'] = demand_shifted.rolling(28, min_periods=1).max().values
    panel_df['roll_zero_count_14'] = (demand_shifted == 0).rolling(14, min_periods=1).sum().values
    for w in [7, 14, 28, 56]:
        panel_df[f'roll_mean_{w}'] = demand_shifted.rolling(w, min_periods=1).mean().values
    for w in [7, 14, 28]:
        panel_df[f'roll_median_{w}'] = demand_shifted.rolling(w, min_periods=1).median().values
        panel_df[f'roll_std_{w}'] = demand_shifted.rolling(w, min_periods=1).std().values
    panel_df['roll_std_56'] = demand_shifted.rolling(56, min_periods=1).std().values
    for w in [56, 84]:
        panel_df[f'roll_max_{w}'] = demand_shifted.rolling(w, min_periods=1).max().values
    rm3 = demand_shifted.rolling(3, min_periods=1).mean().values
    rm14 = demand_shifted.rolling(14, min_periods=1).mean().values
    panel_df['demand_acceleration_3d'] = rm3 / (rm14 + 1e-8)
    panel_df['spike_ratio_28'] = panel_df['roll_max_28'] / (panel_df['roll_mean_28'] + 1e-8)
    panel_df['spike_ratio_56'] = panel_df['roll_max_56'] / (panel_df['roll_mean_56'] + 1e-8)
    panel_df['pct_change_1'] = (panel_df['demand_lag_1'] - panel_df['demand_lag_2']) / (panel_df['demand_lag_2'] + 1e-8)
    panel_df['pct_change_7'] = (panel_df['demand_lag_7'] - panel_df['demand_lag_14']) / (panel_df['demand_lag_14'] + 1e-8)
    last_sale = panel_df['date'].where(demand_shifted > 0)
    last_sale = last_sale.groupby(panel_df[group_cols].apply(tuple, axis=1)).ffill()
    panel_df['days_since_last_sale'] = (panel_df['date'] - last_sale).dt.days.fillna(9999).astype('int16')
    price_s = panel_df.groupby(group_cols)['avg_price'].shift(1)
    panel_df['discount_depth_pct'] = (price_s.rolling(30, min_periods=1).max().values - panel_df['avg_price']) / (price_s.rolling(30, min_periods=1).max().values + 1e-8)
    panel_df['price_momentum'] = panel_df['avg_price'] / (price_s.rolling(14, min_periods=1).mean().values + 1e-8)
    for c in panel_df.select_dtypes(include=['number']).columns:
        panel_df[c] = panel_df[c].replace([np.inf, -np.inf], 0).fillna(0)
    for c in panel_df.select_dtypes(include=['float64']).columns:
        panel_df[c] = panel_df[c].astype('float32')
    del demand_shifted, price_s, rm3, rm14, last_sale; gc.collect()
    print("[FE] Lag+rolling features done")

    # ── Holiday Intensity + Proximity (4 features) ──
    # was: .drop_duplicates() on [[country_code, demand_qty]] which could drop distinct dates
hc = panel_df[panel_df['is_hari_besar'] == 1][['country_code', 'date', 'demand_qty']].drop_duplicates(subset=['country_code', 'date'])
    if not hc.empty:
        hd = hc.groupby('country_code')['demand_qty'].mean().to_dict()
        pd_ = {}
        for code in hd:
            pr = panel_df[(panel_df['country_code'] == code) & (panel_df['is_pre_hari_besar'] == 1)]
            pd_[code] = pr['demand_qty'].mean() if len(pr) > 0 else 1.0
        ir = [{'country_code': c, 'holiday_intensity': hd[c] / pd_.get(c, 1.0) if pd_.get(c, 0) > 0 else 1.0} for c in hd]
        idf = pd.DataFrame(ir)
        panel_df = panel_df.merge(idf, on='country_code', how='left', copy=False)
        del idf
    panel_df['holiday_intensity'] = panel_df.get('holiday_intensity', pd.Series(1.0, index=panel_df.index)).fillna(1.0).astype('float32')
    hc2 = panel_df[panel_df['is_hari_besar'] == 1].groupby('country_code').size()
    low = hc2[hc2 < 5].index
    if len(low) > 0: panel_df.loc[panel_df['country_code'].isin(low), 'holiday_intensity'] = 1.0
    panel_df['holiday_intensity'] = panel_df['holiday_intensity'].clip(upper=HOLIDAY_INTENSITY_CAP)
    del hc, hc2; gc.collect()
    hdates = panel_df[panel_df['is_hari_besar'] == 1][['country', 'date']].drop_duplicates()
    panel_df_sorted = panel_df.sort_values(['country', 'date'], kind='mergesort').reset_index(drop=True)
    hdates = hdates.rename(columns={'date': 'next_holiday'}).sort_values(['country', 'next_holiday'], kind='mergesort')
    if len(hdates) > 0:
        parts = []
        for country, grp in panel_df_sorted.groupby('country', sort=False):
            hgrp = hdates[hdates['country'] == country]
            if hgrp.empty:
                grp['days_to_next_holiday'] = 30
            else:
                m = pd.merge_asof(
                    grp.sort_values('date', kind='mergesort'),
                    hgrp.sort_values('next_holiday', kind='mergesort'),
                    left_on='date',
                    right_on='next_holiday',
                    direction='forward',
                    allow_exact_matches=True,
                )
                grp['days_to_next_holiday'] = (m['next_holiday'] - m['date']).dt.days
            parts.append(grp)
        panel_df_sorted = pd.concat(parts, ignore_index=True)
    else:
        panel_df_sorted['days_to_next_holiday'] = 30
    panel_df_sorted['days_to_next_holiday'] = panel_df_sorted['days_to_next_holiday'].fillna(30).clip(0, 30).astype('int8')
    panel_df_sorted['is_holiday_season'] = ((panel_df_sorted['is_hari_besar'] == 1) | (panel_df_sorted['is_pre_hari_besar'] == 1)).astype('uint8')
    panel_df_sorted['holiday_x_weekend'] = (panel_df_sorted['is_hari_besar'].astype('uint8') & panel_df_sorted['is_weekend'].astype('uint8')).astype('uint8')
    panel_df = panel_df_sorted
    del hdates, panel_df_sorted
    gc.collect()
    print("[FE] Holiday intensity features done")

    # ── Peak Days (1 feature) ──
    daily_total = panel_df.groupby('date')['demand_qty'].sum().sort_values(ascending=False)
    peak_n = max(1, int(len(daily_total) * 0.05))
    panel_df['is_peak_day'] = panel_df['date'].isin(set(daily_total.head(peak_n).index)).astype('uint8')
    print("[FE] Peak day features done")

    # ── Save cache ──
    panel_df.to_parquet(FULL_CACHE)
    print(f"[OK] Full preprocessing cached: {panel_df.shape}")

feat_count = len(panel_df.select_dtypes(include=["number"]).columns)
print(f"[OK] panel_df: {panel_df.shape}, numeric features: {feat_count}")
print(f"[TIME] Feature engineering: {time.time()-_t0:.1f}s")




In [ ]:
# ============================================================
# [CONFIG] Feature group definitions
# ============================================================
SEASONALITY_FEATURES = [
    'day_of_week', 'week_of_year', 'month', 'quarter', 'day_of_month',
    'is_weekend', 'is_month_start', 'is_month_end',
    'days_to_month_end', 'week_of_month',
    'is_month_start_window', 'is_month_end_window',
    'is_hari_besar', 'is_pre_hari_besar',
]
HOLIDAY_INTENSITY_FEATURES = [
    'holiday_intensity', 'days_to_next_holiday',
    'is_holiday_season', 'holiday_x_weekend',
]
TEMPORAL_PEAK_FEATURES = ['is_peak_day']
LAG_ROLL_FEATURES = [
    'demand_lag_1', 'demand_lag_2', 'demand_lag_7', 'demand_lag_14',
    'demand_lag_21', 'demand_lag_28', 'demand_lag_35', 'demand_lag_56', 'demand_lag_84',
    'days_since_last_sale', 'roll_zero_count_14',
    'roll_max_7', 'roll_max_28',
    'roll_mean_7', 'roll_mean_14', 'roll_mean_28', 'roll_mean_56',
    'roll_median_7', 'roll_median_14', 'roll_median_28',
    'roll_std_7', 'roll_std_14', 'roll_std_28', 'roll_std_56',
    'roll_max_56', 'roll_max_84',
    'demand_acceleration_3d',
    'spike_ratio_28', 'spike_ratio_56',
    'pct_change_1', 'pct_change_7',
    'discount_depth_pct', 'price_momentum',
]
ALL_FEATURES = SEASONALITY_FEATURES + HOLIDAY_INTENSITY_FEATURES + TEMPORAL_PEAK_FEATURES + LAG_ROLL_FEATURES

TARGET_COL = 'demand_qty'
PRICE_COL = 'avg_price'
DATE_COL = 'date'
print(f"[CONFIG] Total features: {len(ALL_FEATURES)}")
print(f"[CONFIG] Features: {ALL_FEATURES}")


In [ ]:
# ============================================================
# [PREP] Filter + Time-series folds
# ============================================================
MIN_OBS = 60
item_obs = panel_df.groupby(group_cols).size()
panel_df = panel_df[panel_df.set_index(group_cols).index.isin(item_obs[item_obs >= MIN_OBS].index)].copy()
panel_df = panel_df.sort_values(group_cols + ['date']).reset_index(drop=True)
n_items = panel_df[group_cols].drop_duplicates().shape[0]
print(f"[OK] After filter (min_obs={MIN_OBS}): {panel_df.shape}, items: {n_items}")

def time_series_folds(dates, horizon_days=30, n_splits=3, min_train_days=180):
    dates = np.array(sorted(pd.to_datetime(dates).unique()))
    total_days = len(dates)
    splits = []
    for i in range(n_splits):
        val_end_idx = total_days - (n_splits - i - 1) * horizon_days
        val_start_idx = val_end_idx - horizon_days
        train_end_idx = val_start_idx - 1
        if train_end_idx < min_train_days: continue
        splits.append((dates[train_end_idx], dates[val_start_idx], dates[val_end_idx - 1]))
    return splits

splits = time_series_folds(panel_df[DATE_COL], horizon_days=30, n_splits=3, min_train_days=180)
print(f"[OK] Folds generated: {len(splits)}")
for i, (tr, vs, ve) in enumerate(splits):
    print(f"  Fold {i+1}: train<={tr.date()} | val={vs.date()} to {ve.date()}")


In [ ]:
# ============================================================
# [METRICS] Evaluation functions + asymmetric objective
# ============================================================
def calc_cls(y_true, y_pred, avg_price_arr, margin=0.20):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    avg_price = np.asarray(avg_price_arr, dtype=float)
    cls_val = float(np.sum(np.maximum(y_true - y_pred, 0) * avg_price * margin))
    return cls_val

def smape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float)
    denom = np.abs(y_true) + np.abs(y_pred)
    mask = denom != 0
    return 0.0 if mask.sum() == 0 else float(np.mean(np.abs(y_true[mask] - y_pred[mask]) / denom[mask]) * 100)

def evaluate_prediction(y_true, y_pred, avg_price_arr, y_naive=None):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)))
    smape_val = smape(y_true, y_pred)
    y_true_bin = (np.asarray(y_true) > 0).astype(int)
    y_pred_bin = (np.asarray(y_pred) > 0).astype(int)
    f1_zero = f1_score(y_true_bin, y_pred_bin)
    precision_zero = precision_score(y_true_bin, y_pred_bin, zero_division=0)
    recall_zero = recall_score(y_true_bin, y_pred_bin, zero_division=0)
    cls = calc_cls(y_true, y_pred, avg_price_arr)
    ofr = float(np.minimum(y_true, y_pred).sum() / (y_true.sum() + 1e-8))
    oos_rate = float(np.mean(y_pred < y_true))
    fva = None
    if y_naive is not None:
        baseline_mae = mean_absolute_error(y_true, y_naive)
        if baseline_mae > 0:
            fva = float((baseline_mae - mae) / baseline_mae)
    return {'mae': mae, 'rmse': rmse, 'smape': smape_val, 'f1_zero': f1_zero,
            'precision_zero': precision_zero, 'recall_zero': recall_zero,
            'cls': cls, 'ofr': ofr, 'oos_rate': oos_rate, 'fva': fva}

def asymmetric_obj(alpha=2.0):
    def obj(y_true, y_pred, sample_weight=None):
        residual = y_pred - y_true
        grad = np.where(residual > 0, 2 * residual, 2 * alpha * residual)
        hess = np.where(residual > 0, 2.0, 2.0 * alpha)
        if sample_weight is not None:
            grad *= sample_weight; hess *= sample_weight
        return grad, hess
    return obj

print("[OK] Metrics functions loaded")


In [ ]:
# ============================================================
# [CONFIG] Hyperparameters
# ============================================================
CLF_PARAMS = {
    'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.05,
    'subsample': 0.8, 'colsample_bytree': 0.8,
    'device': DEVICE, 'tree_method': 'hist', 'max_bin': 128,
    'random_state': RANDOM_STATE, 'n_jobs': -1,
}
REG_PARAMS = {
    'n_estimators': 400, 'max_depth': 5, 'learning_rate': 0.05,
    'subsample': 0.7, 'colsample_bytree': 0.8,
    'device': DEVICE, 'tree_method': 'hist', 'max_bin': 128,
    'random_state': RANDOM_STATE, 'n_jobs': -1,
}

# V2 specific
ALPHA_UNDER_TAIL = 200.0
ALPHA_UNDER = ALPHA_UNDER_TAIL
QUANTILE_Q = 0.8
QUANTILE_Q_TOP = 0.9
QUANTILE_Q_PEAK = 0.95
QUANTILE_Q_DYNAMIC_LOW = 0.90

THRESHOLD_GRID = np.round(np.arange(0.05, 0.91, 0.05), 2).tolist()
USE_LOG_TARGET = True
TOP_SEGMENT_PCT = 0.05
SAMPLE_WEIGHT_ALPHA = 4.0
SAMPLE_WEIGHT_CAP = 5.0
HOLIDAY_BOOST = 1.5
PRE_HOLIDAY_BOOST = 0.5
PEAK_DAYS_BOOST = 2.0
HOLIDAY_INTENSITY_HIGH_THRESHOLD = 1.5
HOLIDAY_INTENSITY_CAP = 100.0  # was 10.0
QUANTILE_Q_EXTREME = 0.98  # extra for peak+holiday overlap

print(f"[CONFIG] Device: {DEVICE}")
print(f"[CONFIG] ALPHA_UNDER_TAIL: {ALPHA_UNDER_TAIL}")
print(f"[CONFIG] THRESHOLD_GRID: {THRESHOLD_GRID}")
print(f"[CONFIG] USE_LOG_TARGET: {USE_LOG_TARGET}")



In [ ]:
# ============================================================
# [TRAINING] Twin-XGB Boosted v2 function
# ============================================================
def run_twin_xgb_v2(feature_cols, label, panel_df, splits):
    fold_rows = []
    fold_cache = []
    print(f'[TRAIN] Training mode: device={DEVICE}')
    for fold_idx, (train_end, val_start, val_end) in enumerate(splits, start=1):
        _t = time.time()
        print(f'\n--- Fold {fold_idx}/{len(splits)} ---')

        train_mask = panel_df[DATE_COL] <= train_end
        val_mask = (panel_df[DATE_COL] >= val_start) & (panel_df[DATE_COL] <= val_end)

        train_seg = panel_df.loc[train_mask].groupby(group_cols)[TARGET_COL].sum().sort_values(ascending=False)
        top_n = max(1, int(len(train_seg) * TOP_SEGMENT_PCT))
        top_keys = set(train_seg.head(top_n).index)

        df_tr = panel_df.loc[train_mask, feature_cols + [TARGET_COL, PRICE_COL, 'stock_code', 'country']]
        df_vl = panel_df.loc[val_mask, feature_cols + [TARGET_COL, PRICE_COL, 'stock_code', 'country']]

        X_tr = df_tr[feature_cols].to_numpy(dtype=np.float32, copy=False)
        y_tr = df_tr[TARGET_COL].to_numpy(dtype=np.float32, copy=False)
        X_vl = df_vl[feature_cols].to_numpy(dtype=np.float32, copy=False)
        y_vl = df_vl[TARGET_COL].to_numpy(dtype=np.float32, copy=False)
        price_vl = df_vl[PRICE_COL].to_numpy(dtype=np.float32, copy=False)

        tr_keys = list(zip(df_tr['stock_code'].to_numpy(), df_tr['country'].to_numpy()))
        vl_keys = list(zip(df_vl['stock_code'].to_numpy(), df_vl['country'].to_numpy()))
        vis_top = np.array([k in top_keys for k in vl_keys])

        vis_peak = df_vl['is_peak_day'].to_numpy(dtype=bool)
        vis_holiday = df_vl['is_hari_besar'].to_numpy(dtype=bool) | df_vl['is_pre_hari_besar'].to_numpy(dtype=bool)
        vi = df_vl['holiday_intensity'].to_numpy(dtype=float)
        use_hq = vis_peak | (vi > HOLIDAY_INTENSITY_HIGH_THRESHOLD)

        # ── Classifier ──
        yz = (y_tr > 0).astype(int)
        spw = (len(yz) - yz.sum()) / (yz.sum() + 1e-8)
        clf = xgb.XGBClassifier(**CLF_PARAMS, objective='binary:logistic', scale_pos_weight=spw, feature_names=feature_cols)
        clf.fit(X_tr, yz)
        cal = CalibratedClassifierCV(clf, method='isotonic', cv=3)
        cal.fit(X_tr, yz)
        print(f"  [Fold {fold_idx}] Classifier done")

        # ── Regressors ──
        nm = y_tr > 0
        y_nz = y_tr[nm]
        y_rg = np.log1p(y_nz) if USE_LOG_TARGET else y_nz
        q95 = np.quantile(y_nz, 0.95) if y_nz.size else 0.0
        bw = 1.0 + SAMPLE_WEIGHT_ALPHA * np.minimum(y_nz / (q95 + 1e-8), SAMPLE_WEIGHT_CAP)
        th_arr = df_tr.loc[nm, 'is_hari_besar'].to_numpy(dtype=float)
        tph = df_tr.loc[nm, 'is_pre_hari_besar'].to_numpy(dtype=float)
        tpk = df_tr.loc[nm, 'is_peak_day'].to_numpy(dtype=float)
        em = 1.0 + HOLIDAY_BOOST * th_arr + PRE_HOLIDAY_BOOST * tph + PEAK_DAYS_BOOST * tpk
        sw = bw * em

        rt = xgb.XGBRegressor(**REG_PARAMS, objective='reg:quantileerror', quantile_alpha=QUANTILE_Q_TOP, feature_names=feature_cols)
        rh = xgb.XGBRegressor(**REG_PARAMS, objective='reg:quantileerror', quantile_alpha=QUANTILE_Q_PEAK, feature_names=feature_cols)
        rl = xgb.XGBRegressor(**REG_PARAMS, objective='reg:quantileerror', quantile_alpha=QUANTILE_Q_DYNAMIC_LOW, feature_names=feature_cols)
        rta = xgb.XGBRegressor(**REG_PARAMS, objective=asymmetric_obj(ALPHA_UNDER_TAIL), feature_names=feature_cols)

        re = xgb.XGBRegressor(**REG_PARAMS, objective='reg:quantileerror', quantile_alpha=QUANTILE_Q_EXTREME, feature_names=feature_cols)
        re.fit(X_tr[nm], y_rg, sample_weight=sw)

        rt.fit(X_tr[nm], y_rg, sample_weight=sw)
        rh.fit(X_tr[nm], y_rg, sample_weight=sw)
        rl.fit(X_tr[nm], y_rg, sample_weight=sw)
        rta.fit(X_tr[nm], y_rg)
        print(f"  [Fold {fold_idx}] Regressors done")

        # ── Predict ──
        prob = cal.predict_proba(X_vl)[:, 1]

        def _pred(m, x):
            p = m.predict(x)
            return np.maximum(np.expm1(p) if USE_LOG_TARGET else p, 0)

        p_top = _pred(rt, X_vl)
        p_high = _pred(rh, X_vl)
        p_low = _pred(rl, X_vl)
        p_tail = _pred(rta, X_vl)
        p_extreme = _pred(re, X_vl)

        use_extreme = vis_peak & (vi > HOLIDAY_INTENSITY_HIGH_THRESHOLD)
        prr = np.where(use_extreme, p_extreme,
              np.where(use_hq, p_high,
              np.where(vis_holiday & ~vis_peak, p_low,
              np.where(vis_top, p_top, p_tail))))

        boosted = vis_holiday | vis_peak
        if boosted.any():
            resid = prr[boosted] - y_vl[boosted]
            mr = np.mean(resid)
            if mr < 0:
                factor = 1.0 - mr / (np.mean(y_vl[boosted]) + 1e-8)
                prr[boosted] = prr[boosted] * max(1.0, factor)

        y_naive_v = df_vl['demand_lag_1'].to_numpy(dtype=float, copy=False)
        bm = mean_absolute_error(y_vl, np.nan_to_num(y_naive_v, nan=0.0))
        y_naive_clamped = np.maximum(np.nan_to_num(y_naive_v, nan=0.0), 0)
        cls_naive = calc_cls(y_vl, y_naive_clamped, price_vl)

        best_th, best_cls, best_m = None, None, None
        for threshold in THRESHOLD_GRID:
            pred = prr.copy()
            pred[~vis_peak] = prr[~vis_peak] * (prob[~vis_peak] >= threshold).astype(int)
            m = evaluate_prediction(y_vl, pred, price_vl, y_naive=y_naive_v)
            if best_cls is None or m['cls'] < best_cls:
                best_cls = m['cls']; best_th = threshold; best_m = m

        best_m['threshold'] = best_th; best_m['fold'] = fold_idx
        best_m['cls_naive'] = cls_naive
        best_m['cls_reduction_pct'] = (cls_naive - best_cls) / cls_naive * 100 if cls_naive > 0 else 0.0
        fold_rows.append(best_m)

        fold_cache.append({
            'fold': fold_idx,
            'y_vl': y_vl,
            'price_vl': price_vl,
            'prob': prob,
            'prr': prr,
            'vis_top': vis_top,
            'vis_peak': vis_peak,
            'vis_holiday': vis_holiday,
            'use_hq': use_hq,
            'use_extreme': use_extreme,
        })

        del df_tr, df_vl, X_tr, y_tr, X_vl, y_vl, price_vl, clf, rt, rh, rl, rta, re, cal
        gc.collect()
        cls_red = best_m['cls_reduction_pct']
        elapsed = time.time() - _t
        print(f'  [Fold {fold_idx}] CLS={best_cls:.0f} (naive={cls_naive:.0f}, -{cls_red:.1f}%), OFR={best_m["ofr"]:.3f}, th={best_th}, time={elapsed:.1f}s')

    df = pd.DataFrame(fold_rows)
    agg = df.mean(numeric_only=True).to_dict()
    agg['label'] = label
    return agg, df, fold_cache

print("[OK] run_twin_xgb_v2 function defined")

def find_segment_thresholds(prr, prob, vis_peak, vis_top, y_vl, price_vl, 
                             tail_grid=None, top_grid=None, verbose=False):
    """Grid search untuk best tail + top threshold dengan peak override.
    
    Peak days: no classifier gating (raw reg prediction).
    Tail items: lower threshold. Top items: higher threshold.
    """
    if tail_grid is None:
        tail_grid = [0.01, 0.02, 0.03, 0.05, 0.08, 0.10, 0.15]
    if top_grid is None:
        top_grid = [0.05, 0.08, 0.10, 0.15, 0.20, 0.30]
    
    bm = float(np.mean(np.abs(y_vl)))
    best_cls = None
    best_result = None
    
    for tt in tail_grid:
        for tpt in top_grid:
            pred = prr.copy()
            tail_mask = ~vis_peak & ~vis_top
            top_mask = ~vis_peak & vis_top
            pred[tail_mask] = prr[tail_mask] * (prob[tail_mask] >= tt).astype(int)
            pred[top_mask] = prr[top_mask] * (prob[top_mask] >= tpt).astype(int)
            m = evaluate_prediction(y_vl, pred, price_vl)
            if best_cls is None or m['cls'] < best_cls:
                best_cls = m['cls']
                best_result = (m, pred, tt, tpt)
    
    if verbose and best_result:
        print(f'    Best: tail_th={best_result[2]:.2f}, top_th={best_result[3]:.2f}, '
              f'CLS={best_result[0]["cls"]:.0f}, OFR={best_result[0]["ofr"]:.3f}')
    return best_result  # (metrics, pred, tail_th, top_th)



In [ ]:
# ============================================================
# [TRAIN] Execute Twin-XGB Boosted v2
# ============================================================
_t0 = time.time()
print('[TRAIN] Starting Twin-XGB Boosted v2 (alpha=200, dynamic q)...')
agg_v2, df_v2, fold_cache = run_twin_xgb_v2(ALL_FEATURES, 'boosted_v2', panel_df, splits)
print()
results = pd.DataFrame([agg_v2]).set_index('label').round(4)
print("[RESULT] Aggregated results:")
print(f"[INFO] Cached folds: {len(fold_cache)}")
display(results[['mae','rmse','smape','cls','ofr','oos_rate']])
print(f"[TIME] Total training: {time.time()-_t0:.1f}s")


In [ ]:
# ============================================================
# [EVAL] Fold details
# ============================================================
display(df_v2[['mae','rmse','cls','ofr','threshold','fold']])


In [ ]:
# ============================================================
# [EVAL] Segment evaluation (top vs tail) — with segment thresholds
# ============================================================
_t0 = time.time()
segment_rows = []
for fold_idx, (train_end, val_start, val_end) in enumerate(splits, start=1):
    print(f'  [Segment Eval] Fold {fold_idx}...')
    cache = fold_cache[fold_idx - 1]
    y_vl = cache['y_vl']
    price_vl = cache['price_vl']
    prr = cache['prr']
    prob = cache['prob']
    vis_top = cache['vis_top']
    vis_peak = cache['vis_peak']

    th_global = df_v2[df_v2['fold'] == fold_idx]['threshold'].iloc[0]
    pred_global = prr * (prob >= th_global).astype(int)

    seg_result = find_segment_thresholds(prr, prob, vis_peak, vis_top, y_vl, price_vl, verbose=True)
    if seg_result:
        m_seg, pred_seg, best_tt, best_tpt = seg_result
    else:
        continue

    for seg, mask in [('top', vis_top), ('tail', ~vis_top)]:
        if mask.sum() == 0:
            continue
        mg = evaluate_prediction(y_vl[mask], pred_global[mask], price_vl[mask])
        mg['fold'] = fold_idx; mg['segment'] = seg; mg['method'] = 'global_th'
        segment_rows.append(mg)
        ms = evaluate_prediction(y_vl[mask], pred_seg[mask], price_vl[mask])
        ms['fold'] = fold_idx; ms['segment'] = seg; ms['method'] = 'seg_th'
        segment_rows.append(ms)

seg_df = pd.DataFrame(segment_rows)
print("[EVAL] Segment summary (global_th vs seg_th):")
display(seg_df.groupby(['segment','method']).agg(
    mae=('mae','mean'), rmse=('rmse','mean'),
    cls=('cls','mean'), ofr=('ofr','mean')
).round(4))
print(f"[TIME] Segment eval: {time.time()-_t0:.1f}s")

# Backward compat for summary/viz cells
seg_summary = seg_df[seg_df['method']=='global_th'].groupby('segment').agg(
    mae=('mae','mean'), rmse=('rmse','mean'),
    cls=('cls','mean'), ofr=('ofr','mean')
).reset_index().round(4)


In [ ]:
# ============================================================
# [EVAL] Peak vs Normal day comparison — with peak override
# ============================================================
_t0 = time.time()
peak_rows = []
for fold_idx, (train_end, val_start, val_end) in enumerate(splits, start=1):
    print(f'  [Peak Eval] Fold {fold_idx}...')
    cache = fold_cache[fold_idx - 1]
    use_extreme = cache.get('use_extreme', np.zeros(len(cache['y_vl']), dtype=bool))
    y_vl = cache['y_vl']
    price_vl = cache['price_vl']
    prr = cache['prr']
    prob = cache['prob']
    vis_peak = cache['vis_peak']
    vis_top = cache['vis_top']

    th_global = df_v2[df_v2['fold'] == fold_idx]['threshold'].iloc[0]
    pred_global = prr * (prob >= th_global).astype(int)

    seg_result = find_segment_thresholds(prr, prob, vis_peak, vis_top, y_vl, price_vl)
    if seg_result:
        _, pred_override, _, _ = seg_result
    else:
        continue

    for label, mask in [('peak', vis_peak), ('normal', ~vis_peak)]:
        if mask.sum() == 0:
            continue
        mg = evaluate_prediction(y_vl[mask], pred_global[mask], price_vl[mask])
        mg['fold'] = fold_idx; mg['day_type'] = label; mg['method'] = 'global_th'
        peak_rows.append(mg)
        mo = evaluate_prediction(y_vl[mask], pred_override[mask], price_vl[mask])
        mo['fold'] = fold_idx; mo['day_type'] = label; mo['method'] = 'peak_override'
        peak_rows.append(mo)

p_df = pd.DataFrame(peak_rows)
print("[EVAL] Peak vs Normal (global_th vs peak_override):")
display(p_df.groupby(['day_type','method']).agg(
    mae=('mae','mean'), rmse=('rmse','mean'),
    cls=('cls','mean'), ofr=('ofr','mean')
).round(4))
print(f"[TIME] Peak eval: {time.time()-_t0:.1f}s")

# Backward compat for summary/viz cells
peak_summary = p_df[p_df['method']=='global_th'].groupby('day_type').agg(
    mae=('mae','mean'), rmse=('rmse','mean'),
    cls=('cls','mean'), ofr=('ofr','mean')
).reset_index().round(4)



In [ ]:
# ============================================================
# [EVAL] Quantile bucket evaluation — with segment thresholds
# ============================================================
_t0 = time.time()
pred_rows = []
for fold_idx, (train_end, val_start, val_end) in enumerate(splits, start=1):
    print(f'  [Quantile Eval] Fold {fold_idx}...')
    cache = fold_cache[fold_idx - 1]
    y_vl = cache['y_vl']
    price_vl = cache['price_vl']
    prr = cache['prr']
    prob = cache['prob']
    vis_top = cache['vis_top']
    vis_peak = cache['vis_peak']

    th_global = df_v2[df_v2['fold'] == fold_idx]['threshold'].iloc[0]
    pred_global = prr * (prob >= th_global).astype(int)

    seg_result = find_segment_thresholds(prr, prob, vis_peak, vis_top, y_vl, price_vl)
    if seg_result:
        _, pred_seg, _, _ = seg_result
    else:
        continue

    for method, pred in [('global_th', pred_global), ('seg_th', pred_seg)]:
        nz_mask = y_vl > 0
        if nz_mask.sum() == 0:
            continue
        nz_true = y_vl[nz_mask]
        nz_pred = pred[nz_mask]
        nz_price = price_vl[nz_mask]
        try:
            buckets = pd.qcut(nz_true, q=[0.0, 0.5, 0.8, 0.95, 1.0],
                              labels=['0-50','50-80','80-95','95-100'], duplicates='drop')
        except ValueError:
            continue
        for bucket in buckets.unique():
            mask = buckets == bucket
            m = evaluate_prediction(nz_true[mask], nz_pred[mask], nz_price[mask])
            m['fold'] = fold_idx; m['bucket'] = bucket; m['method'] = method
            pred_rows.append(m)

qdf = pd.DataFrame(pred_rows)
q_pivot = qdf.groupby(['bucket','method'])[['mae','rmse','cls','ofr']].mean().round(2)
print("[EVAL] Quantile bucket (global vs seg_th):")
display(q_pivot)
print(f"[TIME] Quantile eval: {time.time()-_t0:.1f}s")

# Backward compat for summary/viz cells
qdf = qdf[qdf['method']=='seg_th'].copy()
q_pivot = qdf.groupby('bucket')[['mae','rmse','cls','ofr']].mean().round(2)


In [ ]:
# ============================================================
# [ANALYSIS] Holiday Intensity Outlier Analysis
# ============================================================
holiday_countries = panel_df[panel_df['is_hari_besar'] == 1].groupby('country_code').agg(
    n_holidays=('date', 'nunique'),
    mean_intensity=('holiday_intensity', 'mean'),
    total_demand=('demand_qty', 'sum'),
).reset_index().round(2)

print("Countries with < 5 holiday samples (potential outlier):")
few = holiday_countries[holiday_countries['n_holidays'] < 5]
print(few.to_string(index=False) if len(few) > 0 else "(none)")

print(f"\nTotal countries with holidays: {len(holiday_countries)}")
print(f"Mean intensity: {holiday_countries['mean_intensity'].mean():.2f}")
print(f"Max intensity: {holiday_countries['mean_intensity'].max():.2f} (country: {holiday_countries.loc[holiday_countries['mean_intensity'].idxmax(), 'country_code']})")


In [ ]:
# ============================================================
# [VIZ] Performance visualization
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Twin-XGB Boosted v2 — Performance Summary (alpha=200, dynamic q)', fontsize=14, fontweight='bold')

# 1. Fold MAE & RMSE
ax = axes[0, 0]
try:
    ax.plot(df_v2['fold'], df_v2['mae'], 'o-', label='MAE', color='#1f77b4')
    ax.plot(df_v2['fold'], df_v2['rmse'], 's-', label='RMSE', color='#ff7f0e')
    ax.set_xlabel('Fold'); ax.set_ylabel('Error')
    ax.set_title('MAE & RMSE per Fold'); ax.legend(); ax.grid(True, alpha=0.3)
except: ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)

# 2. CLS per Fold
ax = axes[0, 1]
try:
    ax.bar(df_v2['fold'], df_v2['cls'], color=['#2ca02c' if v <= df_v2['cls'].mean() else '#d62728' for v in df_v2['cls']], width=0.5)
    ax.axhline(y=df_v2['cls'].mean(), color='gray', linestyle='--', label=f"Mean: {df_v2['cls'].mean():.0f}")
    ax.set_xlabel('Fold'); ax.set_ylabel('CLS'); ax.set_title('CLS per Fold')
    ax.legend(); ax.grid(True, alpha=0.3)
except: ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)

# 3. Segment OFR
ax = axes[0, 2]
try:
    ax.bar(seg_summary['segment'], seg_summary['ofr'], color=['#1f77b4', '#ff7f0e'], width=0.4)
    ax.axhline(y=0.8, color='red', linestyle='--', label='Target 0.8')
    for i, v in enumerate(seg_summary['ofr']): ax.text(i, v+0.01, f'{v:.3f}', ha='center')
    ax.set_ylabel('OFR'); ax.set_title('OFR by Segment'); ax.legend(); ax.grid(True, alpha=0.3)
except: ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)

# 4. Peak vs Normal OFR
ax = axes[1, 0]
try:
    bars = ax.bar(peak_summary['day_type'], peak_summary['ofr'], color=['#2ca02c', '#d62728'], width=0.4)
    ax.axhline(y=0.8, color='red', linestyle='--', label='Target 0.8')
    for b, v in zip(bars, peak_summary['ofr']): ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01, f'{v:.3f}', ha='center')
    ax.set_ylabel('OFR'); ax.set_title('Peak vs Normal'); ax.legend(); ax.grid(True, alpha=0.3)
except: ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)

# 5. Quantile Bucket OFR
ax = axes[1, 1]
try:
    qp = qdf.groupby('bucket')[['mae','rmse','cls','ofr']].mean().round(2).reset_index()
    colors_b = ['#1f77b4','#ff7f0e','#2ca02c','#d62728']
    bars = ax.bar(qp['bucket'], qp['ofr'], color=colors_b[:len(qp)], width=0.5)
    ax.axhline(y=0.8, color='red', linestyle='--', label='Target 0.8')
    for b, v in zip(bars, qp['ofr']): ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01, f'{v:.2f}', ha='center')
    ax.set_xlabel('Demand Percentile'); ax.set_ylabel('OFR')
    ax.set_title('OFR by Demand Quantile'); ax.legend(); ax.grid(True, alpha=0.3)
except: ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)

# 6. Holiday intensity analysis
ax = axes[1, 2]
try:
    hi = holiday_countries.sort_values('mean_intensity', ascending=False).head(10)
    ax.barh(range(len(hi)), hi['mean_intensity'], color='#1f77b4')
    ax.set_yticks(range(len(hi))); ax.set_yticklabels(hi['country_code'])
    ax.set_xlabel('Mean Intensity'); ax.set_title('Top 10 Holiday Intensity by Country')
    ax.grid(True, alpha=0.3, axis='x'); ax.invert_yaxis()
except: ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
fig.subplots_adjust(top=0.93)
plt.show()
print("[VIZ] Performance plots displayed")


In [ ]:
# ============================================================
# [ANALYSIS] Feature importance (subsample 100k)
# ============================================================
_t0 = time.time()
nz_mask = panel_df[TARGET_COL] > 0
nz_idx = np.where(nz_mask.to_numpy())[0]
rng = np.random.default_rng(RANDOM_STATE)
samp_idx = rng.choice(nz_idx, size=min(100_000, len(nz_idx)), replace=False)
X_s = panel_df.iloc[samp_idx][ALL_FEATURES].to_numpy(dtype=np.float32, copy=False)
y_s = np.log1p(panel_df.iloc[samp_idx][TARGET_COL].to_numpy()) if USE_LOG_TARGET else panel_df.iloc[samp_idx][TARGET_COL].to_numpy()
ip = xgb.XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1,
                       device=DEVICE, tree_method='hist',
                       random_state=RANDOM_STATE, n_jobs=-1,
                       objective='reg:quantileerror', quantile_alpha=QUANTILE_Q_PEAK,
                       feature_names=ALL_FEATURES)
ip.fit(X_s, y_s)
imp = ip.get_booster().get_score(importance_type='gain')
imp_df = pd.DataFrame.from_dict(imp, orient='index', columns=['gain']).sort_values('gain', ascending=False)
imp_df['gain_pct'] = imp_df['gain'] / imp_df['gain'].sum() * 100
# Export feature index map for auditability
feature_map = pd.DataFrame({
    "index": range(len(ALL_FEATURES)),
    "feature_name": ALL_FEATURES,
})
feature_map.to_json("/kaggle/working/feature_index_map.json", orient="records")
print("[AUDIT] Feature index map saved to /kaggle/working/feature_index_map.json")
print("[AUDIT] With feature_names=ALL_FEATURES, get_score() returns proper names, not f0..f51")
print("[FEATURE IMPORTANCE] Top 15:")
display(imp_df.head(15))
print(f"[TIME] Feature importance: {time.time()-_t0:.1f}s")



In [ ]:
# ============================================================
# [SUMMARY] Final Results
# ============================================================
avg_cls_naive = df_v2['cls_naive'].mean()
avg_cls_red = df_v2['cls_reduction_pct'].mean()

print('\n' + '='*60)
print('TWIN-XGB BOOSTED V2 — FINAL SUMMARY')
print('='*60)
print(f'\nParameters:')
print(f'  Device = {DEVICE}')
print(f'  ALPHA_UNDER = {ALPHA_UNDER} (v1 was 50)')
print(f'  THRESHOLD_GRID = {THRESHOLD_GRID[0]:.2f} to {THRESHOLD_GRID[-1]:.2f} (v1 was 0.10)')
print(f'  Dynamic quantile: q=0.95 when holiday_intensity > {HOLIDAY_INTENSITY_HIGH_THRESHOLD}, '
      f'q=0.98 when peak & holiday_intensity > {HOLIDAY_INTENSITY_HIGH_THRESHOLD}')
print()
print('Results:')
print(results)
print(f'CLS reduction vs naive forecast: {avg_cls_red:.1f}% (model CLS={results.loc["boosted_v2","cls"]:.0f}, naive CLS={avg_cls_naive:.0f})')
print()
print('Segment summary:')
print(seg_summary)
print()
print('Peak vs normal:')
print(peak_summary)
print()
print('Quantile bucket:')
print(q_pivot)
print()
print('='*60)
print('[DONE] Notebook completed successfully')
print('='*60)

